# Photon Transfer Curve (PTC)

A PTC measures signal variance against signal level. The slope of the shot-noise
region gives the conversion gain in e-/ADU, the intercept gives the read noise,
and the turnover marks full well.

**The point of this notebook is the fixed frame time.** In normal operation the
camera runs in `AUTO` frame time mode, where the frame period tracks the
exposure (`frame = exposure + 0.1s`). That is wrong for a PTC: if the frame
period changes with every exposure step, then the pixel's reset-to-read interval
changes too, and dark current, persistence and any frame-rate-dependent effect
vary point to point along the curve. Those changes are degenerate with the
signal-dependent variance you are trying to measure.

So we pin the frame time once, up front, and sweep only the exposure time
underneath it.

```
FIXED mode:   |<--------------- 1.1 s frame period --------------->|
              |<-- exposure -->|<-------- reset / idle ----------->|
              |<------- exposure ------->|<---- reset / idle ----->|
              (frame period identical at every point on the curve)
```

The frame time must be at least `exposure + 0.1s`, so a 1.0s maximum exposure
needs a frame time of at least 1.1s. Every exposure shorter than that fits
inside the same frame period.

## 1. Connect

Corrections must be off. Gain, offset and pixel-substitution corrections all
modify pixel values in ways that destroy the variance statistics a PTC depends
on — substitution in particular replaces bad pixels with neighbours, which
correlates pixels that should be independent.

In [ ]:
import os
import time

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

from pirtcam.client import CameraClient

cam = CameraClient("localhost", 5555)
cam.connect()

# All corrections OFF - see above.
cam.set_correction("GAIN", "OFF")
cam.set_correction("OFFSET", "OFF")
cam.set_correction("SUB", "OFF")

cam.set_observer_name("Nate Lourie")
cam.set_object_name("PTC")

print(cam.get_status()["data"]["camera_state"])

## 2. Lab hardware (optional)

Set `USE_LABSERVER = False` to dry-run the notebook without the lab bench, or to
run a PTC on a source you are controlling by other means. With it enabled we
take a matching dark at every exposure level by closing the shutter.

In [ ]:
USE_LABSERVER = True

FILTER_NM = 1050        # bandpass to illuminate through
SETTLE_TIME_S = 2.0     # settle after moving the shutter

if USE_LABSERVER:
    import Pyro5.api as pyro

    labserver = pyro.Proxy("PYRO:LabServer@localhost:50000")
    labserver.select_bandpass(FILTER_NM)
    print(labserver.status()["shutter"]["state"])
else:
    labserver = None
    print("Running without lab hardware - no shutter, no darks.")

## 3. Configure the ramp

The exposure grid is log spaced, which puts most of the points at low signal
where the read-noise floor and the shot-noise knee live, and fewer up near full
well where the curve is a straight line and adds little information.

In [ ]:
EXP_MIN_S = 0.001       # shortest exposure
EXP_MAX_S = 1.0         # longest exposure
N_EXPOSURES = 250       # points along the curve
N_FRAMES = 50           # frames per exposure level (sets variance precision)

# Frame time is held here for the whole ramp. Must be >= EXP_MAX_S + 0.1.
FRAME_TIME_S = 1.1

exptimes = np.geomspace(EXP_MIN_S, EXP_MAX_S, N_EXPOSURES)

assert FRAME_TIME_S >= EXP_MAX_S + 0.1, (
    f"frame time {FRAME_TIME_S}s cannot contain a {EXP_MAX_S}s exposure"
)

# The camera stores timing in clock cycles (15 MHz), so exposures that land on
# the same cycle count are the same exposure. Drop any duplicates rather than
# spending frames measuring the same point twice.
CLOCK_HZ = 15e6
cycles = np.array([int(t * CLOCK_HZ) for t in exptimes])
_, keep = np.unique(cycles, return_index=True)
exptimes = exptimes[np.sort(keep)]

print(f"{len(exptimes)} distinct exposure levels")
print(f"  {exptimes[0]*1e3:.3f} ms -> {exptimes[-1]*1e3:.1f} ms")
print(f"  step ratio {exptimes[1]/exptimes[0]:.4f} ({(exptimes[1]/exptimes[0]-1)*100:.2f}% per step)")
print(f"  frame time fixed at {FRAME_TIME_S}s")

## 4. Time budget — read this before running

Because the frame time is fixed, **a 1 ms exposure costs exactly as much wall
clock as a 1 s exposure**. That is the price of a clean PTC, and it makes the
total runtime a simple product. Check the number below before you start.

In [ ]:
# Per-frame software overhead on top of the camera frame period: serial setup,
# arming the grabber, and the per-frame header queries. Measured empirically;
# adjust once you have timed a real run.
OVERHEAD_PER_FRAME_S = 2.0

per_frame_s = FRAME_TIME_S + OVERHEAD_PER_FRAME_S
sets_per_level = 2 if USE_LABSERVER else 1   # lights, plus darks if shuttered

total_frames = len(exptimes) * N_FRAMES * sets_per_level
total_s = total_frames * per_frame_s

print(f"frames:          {total_frames:,}")
print(f"per frame:       {per_frame_s:.1f}s  ({FRAME_TIME_S}s frame + {OVERHEAD_PER_FRAME_S}s overhead)")
print(f"TOTAL:           {total_s/3600:.1f} hours")
print()
print(f"  camera time:   {total_frames*FRAME_TIME_S/3600:.1f} h")
print(f"  overhead:      {total_frames*OVERHEAD_PER_FRAME_S/3600:.1f} h  <- dominates")

If that number is too large, the two knobs that matter are:

**`N_FRAMES` — this is the expensive one.** The fractional uncertainty on a
variance estimated from `N` frames is roughly `1/sqrt(2(N-1))`: 50 frames buys
about 10%, 20 frames about 16%, 10 frames about 24%. Runtime is linear in
`N_FRAMES`, so going 50 → 20 cuts the run by 60% and costs you a factor 1.6 in
variance precision. Since you then fit a line through hundreds of exposure
levels, the fit absorbs most of that.

**`N_EXPOSURES`** — 250 points is a very finely sampled curve. A PTC is
fundamentally a two-parameter fit (gain and read noise) plus a full-well
turnover; 60–80 log-spaced points resolve all three comfortably.

A `N_EXPOSURES = 80`, `N_FRAMES = 20` run covers the same dynamic range in about
a tenth of the time. Worth doing first as a shakedown, then running the full
grid overnight once you know the illumination level is right.

The overhead line above is worth noting too: at 2.0s per frame it costs nearly
twice the actual camera time. Most of it is the ~20 register queries done per
frame to build the FITS header, most of which are static across a stack.

## 5. Pin the frame time

This is the step that makes it a PTC rather than a ramp. After this call the
camera is in `FIXED` mode and `set_exposure()` will no longer move the frame
period.

If an exposure ever exceeds `FRAME_TIME_S - 0.1`, `set_exposure()` fails loudly
and leaves the camera alone rather than silently clipping the integration — a
quietly shortened exposure would bend the curve with no visible symptom.

In [ ]:
resp = cam.set_frame_time(FRAME_TIME_S)
print(resp)

status = cam.get_status()["data"]
print(f"\nframe time mode: {status['frame_time_mode']}")
print(f"frame time:      {status['frame_time']:.6f} s")
print(f"exposure:        {status['exposure']:.6f} s")

assert status["frame_time_mode"] == "FIXED", "frame time is not pinned!"

### Sanity check: exposure moves, frame time does not

Worth running once. If `frame time` changes in this loop, the ramp below is not
a PTC and the results will be wrong.

In [ ]:
for test_exp in [0.001, 0.1, 1.0]:
    cam.set_exposure(test_exp, wait=True)
    s = cam.get_status()["data"]
    print(f"exposure {s['exposure']:.6f}s  ->  frame time {s['frame_time']:.6f}s")

assert abs(cam.get_status()["data"]["frame_time"] - FRAME_TIME_S) < 1e-6
print("\nFrame time held constant across the full exposure range.")

## 6. Run the ramp

One stacked FITS per exposure level per shutter state. Stacking keeps the file
count manageable and amortises the file write.

**Both waits below are required, and both are off by default in the client:**

- `set_exposure(..., wait=True)` blocks until the new timing has settled (three
  frame periods). Without it the first frames of the stack can be taken while
  the camera is still changing state.
- `capture_frames(..., wait_for_completion=True)` blocks until the stack is
  saved. Without it `capture_frames` returns as soon as the server acknowledges
  the command, and the loop would set the *next* exposure while the current
  stack is still being taken — silently mislabelling frames.

Note that the wait loop inside `capture_frames` currently warns but does not
give up if a capture stalls, so a hung capture will block the notebook rather
than raise. Keep an eye on the first few levels.

The loop is restartable: it skips any level whose output already exists, so an
interrupted overnight run resumes by re-executing the cell.

In [ ]:
base_dir = os.path.join(os.path.expanduser("~"), "data", "ptc")
date_prefix = time.strftime("%Y%m%d")
run_dir = os.path.join(base_dir, f"{date_prefix}_{FILTER_NM}")
os.makedirs(run_dir, exist_ok=True)

print(f"Saving to {run_dir}")


def build_header(exptime, shutter_status, kind):
    hdr = [
        ("REXPTIME", float(exptime), "Requested exposure time in seconds"),
        ("RFRMTIME", float(FRAME_TIME_S), "Requested frame time in seconds"),
        ("PTCKIND", kind, "PTC frame kind: light or dark"),
        ("PTCNFRM", int(N_FRAMES), "Frames in this stack"),
        ("SHUTTER", shutter_status, "Shutter status: open or closed"),
    ]
    if USE_LABSERVER:
        hdr += [
            ("FILTER", FILTER_NM, "Filter band center in nm"),
            ("FW1FILT", labserver.wheel_status("fw1")["current_filter"], "Filter in fw1"),
            ("FW2FILT", labserver.wheel_status("fw2")["current_filter"], "Filter in fw2"),
            ("FW3FILT", labserver.wheel_status("fw3")["current_filter"], "Filter in fw3"),
            ("FW4FILT", labserver.wheel_status("fw4")["current_filter"], "Filter in fw4"),
        ]
    return hdr


def set_shutter(state):
    if not USE_LABSERVER:
        return "unknown"
    labserver.shutter(state)
    time.sleep(SETTLE_TIME_S)
    return labserver.status()["shutter"]["state"]


kinds = ["dark", "light"] if USE_LABSERVER else ["light"]

t_start = time.time()
for i, exptime in enumerate(exptimes):
    # wait=True: block until the new timing has settled before capturing.
    cam.set_exposure(float(exptime), wait=True)

    for kind in kinds:
        out_dir = os.path.join(run_dir, kind)
        os.makedirs(out_dir, exist_ok=True)
        fname = f"ptc_{kind}_{i:04d}_{exptime*1e3:.4f}ms.fits"

        if os.path.exists(os.path.join(out_dir, fname)):
            continue   # already done - restartable

        shutter_status = set_shutter("close" if kind == "dark" else "open")

        cam.set_save_path(out_dir)
        cam.capture_frames(
            N_FRAMES,
            stack=True,
            filename=fname,
            headers=build_header(exptime, shutter_status, kind),
            show_progress=False,
            # wait_for_completion=True: do not advance to the next exposure
            # until this stack is on disk.
            wait_for_completion=True,
        )

    elapsed = time.time() - t_start
    frac = (i + 1) / len(exptimes)
    eta = elapsed / frac - elapsed
    print(
        f"[{i+1:3d}/{len(exptimes)}] {exptime*1e3:8.3f} ms   "
        f"elapsed {elapsed/3600:5.2f} h   eta {eta/3600:5.2f} h",
        end="\r",
    )

print(f"\nDone in {(time.time()-t_start)/3600:.2f} hours.")

## 7. Release the frame time

Put the camera back in `AUTO` so normal observing is not left with a 1.1s floor
on every exposure.

In [ ]:
print(cam.set_frame_time_mode("AUTO"))
s = cam.get_status()["data"]
print(f"mode {s['frame_time_mode']}, exposure {s['exposure']:.6f}s, frame {s['frame_time']:.6f}s")

if USE_LABSERVER:
    labserver.shutter("close")

## 8. Analysis

With `N_FRAMES` frames at each level the variance is taken **per pixel across
the stack**, then averaged over pixels. Temporal variance is the right estimator
here: fixed-pattern non-uniformity is constant frame to frame, so it cancels
without needing difference images.

- signal `S` = mean(light) − mean(dark), in ADU
- variance `V` = median over pixels of the per-pixel temporal variance
- gain `g` = 1/slope of `V` vs `S` in the shot-noise region, in e-/ADU
- read noise = `sqrt(intercept) * g`, in e-

In [ ]:
def stack_stats(path, region=np.s_[:, :]):
    """Mean frame and per-pixel temporal variance from a stacked FITS cube."""
    with fits.open(path) as hdul:
        cube = hdul[0].data.astype(np.float64)
        exptime = hdul[0].header.get("EXPTIME", np.nan)
    cube = cube[(slice(None),) + region]
    return exptime, cube.mean(axis=0), cube.var(axis=0, ddof=1)


# Central region avoids edge effects and reference pixels.
REGION = np.s_[100:-100, 100:-100]

light_dir = os.path.join(run_dir, "light")
dark_dir = os.path.join(run_dir, "dark")

records = []
for fname in sorted(os.listdir(light_dir)):
    if not fname.endswith(".fits"):
        continue
    exptime, light_mean, light_var = stack_stats(os.path.join(light_dir, fname), REGION)

    dark_path = os.path.join(dark_dir, fname.replace("_light_", "_dark_"))
    if os.path.exists(dark_path):
        _, dark_mean, dark_var = stack_stats(dark_path, REGION)
    else:
        dark_mean = np.zeros_like(light_mean)
        dark_var = np.zeros_like(light_var)

    records.append(
        {
            "exptime": exptime,
            "signal": np.median(light_mean - dark_mean),
            # Subtracting the dark's temporal variance removes the read-noise
            # contribution that both share, leaving the photon term.
            "variance": np.median(light_var) - np.median(dark_var),
            "read_var": np.median(dark_var),
        }
    )

signal = np.array([r["signal"] for r in records])
variance = np.array([r["variance"] for r in records])
exptime = np.array([r["exptime"] for r in records])
print(f"{len(records)} exposure levels loaded")

In [ ]:
# Fit the shot-noise region. Adjust the bounds to exclude the read-noise floor
# at the bottom and the full-well turnover at the top.
FIT_LO, FIT_HI = 500, 20000     # ADU

mask = (signal > FIT_LO) & (signal < FIT_HI)
slope, intercept = np.polyfit(signal[mask], variance[mask], 1)

gain = 1.0 / slope                       # e-/ADU
read_noise_adu = np.sqrt(max(intercept, 0))
read_noise_e = read_noise_adu * gain

print(f"gain:       {gain:.3f} e-/ADU")
print(f"read noise: {read_noise_adu:.2f} ADU  =  {read_noise_e:.2f} e-")
print(f"fit over {mask.sum()} of {len(signal)} levels")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.loglog(signal, variance, ".", ms=4, label="measured")
xs = np.geomspace(max(signal[mask].min(), 1), signal[mask].max(), 100)
ax.loglog(xs, slope * xs + intercept, "-", lw=1.5, label=f"fit: g={gain:.3f} e-/ADU")
ax.axvspan(FIT_LO, FIT_HI, alpha=0.1, color="green", label="fit region")
ax.set_xlabel("signal [ADU]")
ax.set_ylabel("variance [ADU$^2$]")
ax.set_title("Photon transfer curve")
ax.legend()
ax.grid(alpha=0.3, which="both")

ax = axes[1]
ax.loglog(exptime, signal, ".", ms=4)
ax.set_xlabel("exposure time [s]")
ax.set_ylabel("signal [ADU]")
ax.set_title("Linearity (fixed frame time)")
ax.grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

### Reading the curve

- **Slope 1 in log-log** is the shot-noise region: variance ∝ signal, this is
  where the gain fit is valid.
- **Flat at the bottom left** is the read-noise floor, where variance stops
  depending on signal.
- **Turnover at the top right** is the onset of full well; the fit region must
  stop below it.
- **Departures from slope 1 in the middle** usually mean a correction is still
  enabled, or the illumination drifted during the run. Check `GAINCOR`,
  `OFFCOR` and `SUBCOR` in the headers — all three should read 0.

If the linearity panel is not a straight line through the origin, the
illumination was not stable across the run, and the PTC is measuring the source
rather than the detector.